# Decodificación de variables categóricas ENIGH

Esta revisión parte de `data/interim/revision_2/`, usa los PDFs oficiales `doc_2018.pdf`, `doc_2020.pdf`, `doc_2022.pdf` y `doc_2024.pdf`, y genera `data/interim/revision_3/` con columnas descriptivas `_desc` únicamente cuando los códigos observados tienen mapping oficial por año.


## Método

- Las fichas de la sección 2.3 se leen con `pdfplumber` y se extraen filas `Valor -> Etiqueta`.
- Las variables que remiten a catálogos de la sección 2.4 se enlazan sólo cuando el catálogo está localizado en el mismo PDF.
- La validación compara códigos observados contra códigos documentados por año, tabla y variable.
- Una variable se decodifica sólo si sus códigos observados quedan cubiertos; la variable original se conserva.


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
REV3 = ROOT / "data" / "interim" / "revision_3"

catalogo = pd.read_csv(REV3 / "catalogo_categoricas_enigh.csv")
estabilidad = pd.read_csv(REV3 / "matriz_estabilidad_categoricas.csv")
validacion = pd.read_csv(REV3 / "validacion_mappings_categoricas.csv")
inventario = pd.read_csv(REV3 / "inventario_variables_revision_3.csv")

print("Catalogo:", catalogo.shape)
print("Estabilidad:", estabilidad.shape)
print("Validacion:", validacion.shape)
print("Inventario:", inventario.shape)


Catalogo: (4406, 12)
Estabilidad: (688, 8)
Validacion: (1296, 11)
Inventario: (558, 6)


In [2]:
catalogo.query("tabla == 'poblacion' and variable in ['sexo','alfabetism','asis_esc','edo_conyug','nivelaprob','parentesco','lenguaind','residencia']").head(80)


,anio,tabla,variable,tipo_documentado,codigo,etiqueta,rango,definicion,catalogo_referencia,orden_categoria,fuente,origen_mapping
281,2018,poblacion,parentesco,C (3),101,Jefe(a),Catálogo de parentesco,"conyugal, por consanguinidad, adopción,",catalogo_parentesco,NaN,doc_2018.pdf p.179,catalogo_externo_pdf
282,2018,poblacion,parentesco,C (3),102,Persona sola,Catálogo de parentesco,"conyugal, por consanguinidad, adopción,",catalogo_parentesco,NaN,doc_2018.pdf p.179,catalogo_externo_pdf
283,2018,poblacion,parentesco,C (3),201,"Esposo(a), compañero(a), cónyuge, pareja, mari...",Catálogo de parentesco,"conyugal, por consanguinidad, adopción,",catalogo_parentesco,NaN,doc_2018.pdf p.179,catalogo_externo_pdf
284,2018,poblacion,parentesco,C (3),202,Concubino(a),Catálogo de parentesco,"conyugal, por consanguinidad, adopción,",catalogo_parentesco,NaN,doc_2018.pdf p.179,catalogo_externo_pdf
285,2018,poblacion,parentesco,C (3),203,Amasio(a),Catálogo de parentesco,"conyugal, por consanguinidad, adopción,",catalogo_parentesco,NaN,doc_2018.pdf p.179,catalogo_externo_pdf
...,...,...,...,...,...,...,...,...,...,...,...,...
480,2018,poblacion,nivelaprob,C (1),6,Carrera técnica o comercial,"{0,...,9}",integrante del hogar de 3 o más años dentro de...,NaN,6,doc_2018.pdf p.74,ficha_variable
481,2018,poblacion,nivelaprob,C (1),7,Profesional,"{0,...,9}",integrante del hogar de 3 o más años dentro de...,NaN,7,doc_2018.pdf p.74,ficha_variable
482,2018,poblacion,nivelaprob,C (1),8,Maestría,"{0,...,9}",integrante del hogar de 3 o más años dentro de...,NaN,8,doc_2018.pdf p.74,ficha_variable
483,2018,poblacion,nivelaprob,C (1),9,Doctorado,"{0,...,9}",integrante del hogar de 3 o más años dentro de...,NaN,9,doc_2018.pdf p.74,ficha_variable


In [3]:
pd.read_csv(REV3 / 'muestra_poblacion_prioritaria.csv')


,anio,tabla,variable,observados,documentados,mapeados,sin_mapping,codigos_sin_mapping,documentados_no_observados,normalizaciones_para_mapping,tiene_mapping
0,2018,poblacion,alfabetism,2,2,2,0,NaN,0,NaN,True
1,2018,poblacion,asis_esc,2,2,2,0,NaN,0,NaN,True
2,2018,poblacion,edo_conyug,6,6,6,0,NaN,0,NaN,True
3,2018,poblacion,etnia,2,2,2,0,NaN,0,NaN,True
4,2018,poblacion,hablaind,2,2,2,0,NaN,0,NaN,True
5,2018,poblacion,lenguaind,84,0,0,84,0111; 0113; 0114; 0115; 0121; 0131; 0200; 0211...,0,NaN,False
6,2018,poblacion,nivelaprob,10,10,10,0,NaN,0,NaN,True
7,2018,poblacion,parentesco,34,63,34,0,NaN,29,NaN,True
8,2018,poblacion,residencia,34,34,34,0,NaN,0,NaN,True
9,2018,poblacion,sexo,2,2,2,0,NaN,0,NaN,True


In [4]:
estabilidad['estado'].value_counts(dropna=False)


estado
no disponible en todos los años    395
estable                            207
categorías diferentes               86
Name: count, dtype: int64

In [5]:
validacion.query('sin_mapping > 0').head(60)


,anio,tabla,variable,observados,documentados,mapeados,sin_mapping,codigos_sin_mapping,documentados_no_observados,normalizaciones_para_mapping,tiene_mapping
22,2018,hogares,anio_aspir,37,1,0,37,00; 01; 02; 03; 04; 05; 06; 07; 08; 09; 10; 11...,1,NaN,True
23,2018,hogares,anio_auto,52,1,1,51,00; 01; 02; 03; 04; 05; 06; 07; 08; 09; 10; 11...,0,NaN,True
24,2018,hogares,anio_bici,57,1,1,56,00; 01; 02; 03; 04; 05; 06; 07; 08; 09; 10; 11...,0,NaN,True
25,2018,hogares,anio_canoa,32,1,0,32,00; 01; 02; 03; 04; 05; 06; 07; 08; 09; 10; 11...,1,NaN,True
26,2018,hogares,anio_carret,36,1,1,35,00; 01; 02; 03; 04; 05; 06; 07; 08; 09; 10; 11...,0,NaN,True
27,2018,hogares,anio_compu,29,1,1,28,00; 01; 02; 03; 04; 05; 06; 07; 08; 09; 10; 11...,0,NaN,True
28,2018,hogares,anio_dvd,32,1,0,32,00; 01; 02; 03; 04; 05; 06; 07; 08; 09; 10; 11...,1,NaN,True
29,2018,hogares,anio_ester,49,1,1,48,00; 01; 02; 03; 04; 05; 06; 07; 08; 09; 10; 11...,0,NaN,True
30,2018,hogares,anio_estuf,66,1,1,65,00; 01; 02; 03; 04; 05; 06; 07; 08; 09; 10; 11...,0,NaN,True
31,2018,hogares,anio_impre,27,1,0,27,00; 01; 02; 03; 04; 05; 06; 07; 08; 09; 10; 11...,1,NaN,True


In [6]:
inventario.query("decodificada == 'Sí'").head(80)


,tabla,variable,tipo_conceptual,decodificada,estable_4_anios,observacion
4,concentradohogar,tam_loc,categórica ordinal,Sí,estable,Decodificada con mapping oficial por año.
5,concentradohogar,est_socio,categórica ordinal,Sí,categorías diferentes,Decodificada con mapping oficial por año.
9,concentradohogar,clase_hog,categórica nominal,Sí,categorías diferentes,Decodificada con mapping oficial por año.
10,concentradohogar,sexo_jefe,categórica nominal,Sí,estable,Decodificada con mapping oficial por año.
12,concentradohogar,educa_jefe,categórica nominal,Sí,estable,Decodificada con mapping oficial por año.
...,...,...,...,...,...,...
300,poblacion,gradoaprob,categórica ordinal,Sí,categorías diferentes,Decodificada con mapping oficial por año.
301,poblacion,antec_esc,categórica nominal,Sí,categorías diferentes,Decodificada con mapping oficial por año.
302,poblacion,residencia,código de catálogo,Sí,categorías diferentes,Decodificada con mapping oficial por año.
303,poblacion,edo_conyug,categórica nominal,Sí,categorías diferentes,Decodificada con mapping oficial por año.


In [7]:
pob = pd.read_csv(REV3 / 'poblacion_decodificada_2018_2024.csv.gz', nrows=20)
pob[[c for c in ['anio','sexo','sexo_desc','alfabetism','alfabetism_desc','asis_esc','asis_esc_desc','edo_conyug','edo_conyug_desc','nivelaprob','nivelaprob_desc'] if c in pob.columns]].head()


,anio,sexo,sexo_desc,alfabetism,alfabetism_desc,asis_esc,asis_esc_desc,edo_conyug,edo_conyug_desc,nivelaprob,nivelaprob_desc
0,2018,1,Hombre,1,Sí,2,No,2.0,Está casado(a),2,Primaria
1,2018,2,Mujer,1,Sí,2,No,2.0,Está casado(a),2,Primaria
2,2018,1,Hombre,1,Sí,2,No,6.0,Está soltero(a),4,Preparatoria o bachillerato
3,2018,1,Hombre,1,Sí,2,No,2.0,Está casado(a),9,Doctorado
4,2018,2,Mujer,1,Sí,2,No,2.0,Está casado(a),7,Profesional
